# Mentor Clustering Model - Edu Growth
Risk model (`risk_model.pkl`) pehle se bana hua hai, wo 0/1 deta hai (1 = risk). Us output ko use karke teachers ko judge karna, students pe KMeans lagana aur mentor assign karna.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

In [ ]:
subjects = ['coa', 'maths4', 'dstl', 'ds', 'python', 'cyber']
ST_MAX = 30    # dataset me ST out of 30 tha, alag ho to yahi badal dena
UNIT_MAX = 20
PUT_MAX = 100
BOTTOM_PCT = 40    # isse kam percent = bottomer
TOP_PCT = 75       # isse zyada percent = topper
# risk model me feature names save nahi hue to ye columns use honge (jin pe train hua tha wahi)
RISK_COLS_FALLBACK = ['coa_st1_marks', 'ds_st1_marks', 'overall_attendance_pct', 'previous_cgpa']

In [ ]:
def find_file(options):
    for p in options:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'koi bhi path nahi mila: {options}')
data_path = find_file(['cleaned_edu_growth_dataset.csv', '../data/processed/cleaned_edu_growth_dataset.csv', 'data/processed/cleaned_edu_growth_dataset.csv'])
risk_path = find_file(['risk_model.pkl', 'models/risk_model.pkl', '../models/risk_model.pkl', 'artifacts/risk_model.pkl', '../artifacts/risk_model.pkl'])
df = pd.read_csv(data_path)
print(df.shape)
df[['roll_no', 'full_name', 'class_section', 'previous_cgpa']].head()

In [ ]:
# jo columns chahiye wo sab hain ya nahi, aur outlier ho to range ke andar kar do
unit_cols = [f'{s}_unit_{u}_marks' for s in subjects for u in range(1, 6)]
put_cols = [f'{s}_put_marks' for s in subjects]
need = ['roll_no', 'full_name', 'class_section', 'overall_attendance_pct', 'previous_cgpa'] + unit_cols + put_cols
for s in subjects:
    need += [f'{s}_st1_marks', f'{s}_st2_marks', f'{s}_assignment_delay_hours']
missing_cols = [c for c in need if c not in df.columns]
assert len(missing_cols) == 0, missing_cols
num_cols = [c for c in need if c not in ('roll_no', 'full_name', 'class_section')]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[unit_cols] = df[unit_cols].clip(0, UNIT_MAX)
df[put_cols] = df[put_cols].clip(0, PUT_MAX)
print('columns theek hain')

In [ ]:
# dataset me teacher column nahi hai, to section ke hisab se teacher tay kar rahe hain (naam apne actual faculty ke daal lena)
teachers = {
    'coa': {'Dr. Sharma': ['CSE-A', 'CSE-B', 'IT-A'], 'Prof. Verma': ['CSE-C', 'IT-B'], 'Dr. Ananya': ['CS-AI', 'CS-DS']},
    'maths4': {'Dr. Rajesh': ['CSE-A', 'CSE-C', 'CS-AI'], 'Prof. Sneha': ['CSE-B', 'IT-B'], 'Dr. Mehta': ['IT-A', 'CS-DS']},
    'dstl': {'Prof. Gupta': ['CSE-A', 'IT-B'], 'Dr. Iyer': ['CSE-B', 'CSE-C', 'CS-DS'], 'Prof. Khan': ['IT-A', 'CS-AI']},
    'ds': {'Dr. Singhal': ['CSE-A', 'CSE-B'], 'Prof. Mishra': ['CSE-C', 'IT-A', 'CS-AI'], 'Dr. Kapoor': ['IT-B', 'CS-DS']},
    'python': {'Prof. Joshi': ['CSE-A', 'CS-DS'], 'Dr. Nair': ['CSE-B', 'CSE-C', 'IT-A'], 'Prof. Bansal': ['IT-B', 'CS-AI']},
    'cyber': {'Dr. Saxena': ['CSE-A', 'CSE-C'], 'Prof. Tiwari': ['CSE-B', 'IT-B', 'CS-AI'], 'Dr. Malhotra': ['IT-A', 'CS-DS']},
}
section = df['class_section'].astype(str).str.strip().str.upper()
for s in subjects:
    sec_to_teacher = {sec: t for t, secs in teachers[s].items() for sec in secs}
    df[f'{s}_teacher'] = section.map(sec_to_teacher).fillna('Unknown')
print((df['coa_teacher'] == 'Unknown').sum(), 'bachon ka section match nahi hua')

## 1. Risk model (0 / 1)

In [ ]:
risk_model = joblib.load(risk_path)
if hasattr(risk_model, 'feature_names_in_'):
    risk_cols = list(risk_model.feature_names_in_)
else:
    risk_cols = RISK_COLS_FALLBACK
miss = [c for c in risk_cols if c not in df.columns]
assert len(miss) == 0, f'risk model ke ye columns dataset me nahi hain: {miss}'
# 0 = no risk, 1 = risk
df['risk'] = risk_model.predict(df[risk_cols].fillna(0)).astype(int)
print(df['risk'].value_counts())
print(round(df['risk'].mean() * 100, 1), '% bache risk me hain')

## 2. KMeans clustering

In [ ]:
# progress = ST2 % - ST1 %, plus matlab improve ho raha hai, minus matlab gir raha hai
df['avg_unit_pct'] = (df[unit_cols] / UNIT_MAX * 100).mean(axis=1)
df['avg_put_pct'] = (df[put_cols] / PUT_MAX * 100).mean(axis=1)
df['avg_delay_hrs'] = df[[f'{s}_assignment_delay_hours' for s in subjects]].mean(axis=1)
st1 = df[[f'{s}_st1_marks' for s in subjects]].clip(0, ST_MAX) / ST_MAX * 100
st2 = df[[f'{s}_st2_marks' for s in subjects]].clip(0, ST_MAX) / ST_MAX * 100
df['avg_progress'] = st2.values.mean(axis=1) - st1.values.mean(axis=1)
df[['avg_unit_pct', 'avg_put_pct', 'avg_progress', 'avg_delay_hrs']].describe().round(1)

In [ ]:
cluster_cols = ['avg_unit_pct', 'avg_put_pct', 'avg_progress', 'overall_attendance_pct', 'previous_cgpa', 'avg_delay_hrs', 'risk']
scaler = StandardScaler()
X = scaler.fit_transform(df[cluster_cols])
# README me 4 cohort likhe hain isliye k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)
summary = df.groupby('cluster')[cluster_cols].mean().round(1)
summary['students'] = df['cluster'].value_counts().sort_index()
summary

In [ ]:
# marks sabse kam wala = Critical Need, sabse zyada = High Achiever, beech ke do me jiska progress kam wo Slumper
low = int(summary['avg_unit_pct'].idxmin())
high = int(summary['avg_unit_pct'].idxmax())
mid = [int(c) for c in summary.index if c not in (low, high)]
slump = int(summary.loc[mid, 'avg_progress'].idxmin())
cluster_names = {low: 'Critical Need', high: 'High Achiever', slump: 'Slumper'}
for c in mid:
    if c != slump:
        cluster_names[c] = 'Consistent Performer'
print(cluster_names)
df['cluster_name'] = df['cluster'].map(cluster_names)
# 1 = sabse urgent
df['priority'] = df['cluster_name'].map({'Critical Need': 1, 'Slumper': 2, 'Consistent Performer': 3, 'High Achiever': 4})
df['cluster_name'].value_counts()

## 3. Teacher efficacy (unit wise)

In [ ]:
# bottomer = unit me 40% se kam ya risk wala bacha, topper = unit me 75%+ aur risk nahi
rows = []
for s in subjects:
    for u in range(1, 6):
        pct = df[f'{s}_unit_{u}_marks'] / UNIT_MAX * 100
        tmp = pd.DataFrame({'teacher': df[f'{s}_teacher'], 'bottom': (pct < BOTTOM_PCT) | (df['risk'] == 1), 'top': (pct >= TOP_PCT) & (df['risk'] == 0)})
        tmp = tmp[tmp['teacher'] != 'Unknown']
        g = tmp.groupby('teacher').agg(total=('bottom', 'size'), bottomers=('bottom', 'sum'), toppers=('top', 'sum')).reset_index()
        g['subject'] = s
        g['unit'] = u
        rows.append(g)
t_eff = pd.concat(rows, ignore_index=True)
t_eff.head()

In [ ]:
# kam bottomers aur zyada toppers = high efficacy (score 1 se 10)
t_eff['efficacy'] = (5 + 5 * t_eff['toppers'] / t_eff['total'] - 5 * t_eff['bottomers'] / t_eff['total']).clip(1, 10).round(2)
t_eff = t_eff.sort_values(['subject', 'unit', 'efficacy'], ascending=[True, True, False]).reset_index(drop=True)
t_eff['rank'] = t_eff.groupby(['subject', 'unit']).cumcount() + 1
t_eff[t_eff['rank'] == 1][['subject', 'unit', 'teacher', 'bottomers', 'toppers', 'efficacy']].head(15)

## 4. Mentor assignment

In [ ]:
# har bache ka sabse weak (subject, unit)
unit_pct = df[unit_cols] / UNIT_MAX * 100
weak_col = unit_pct.idxmin(axis=1)
df['weak_pct'] = unit_pct.min(axis=1)
df['weak_subject'] = weak_col.str.split('_').str[0]
df['weak_unit'] = weak_col.str.extract(r'unit_(\d)_marks')[0].astype(int)
# mentor chahiye: risk model ne jinhe 1 diya + jo Slumper cluster me hain (progress gir raha hai)
df['needs_mentor'] = (df['risk'] == 1) | (df['cluster_name'] == 'Slumper')
print(df['needs_mentor'].sum(), 'bachon ko mentor chahiye, total', len(df))
df[['roll_no', 'weak_subject', 'weak_unit', 'weak_pct', 'needs_mentor']].head()

In [ ]:
# mentor = us unit ka highest efficacy teacher, par agar bache ka apna teacher hi best hai to second best do
best1 = t_eff[t_eff['rank'] == 1].set_index(['subject', 'unit'])
best2 = t_eff[t_eff['rank'] == 2].set_index(['subject', 'unit'])
key = pd.MultiIndex.from_arrays([df['weak_subject'], df['weak_unit']])
df['best_teacher'] = best1['teacher'].reindex(key).values
df['best_eff'] = best1['efficacy'].reindex(key).values
df['second_teacher'] = best2['teacher'].reindex(key).values
df['second_eff'] = best2['efficacy'].reindex(key).values
df['own_teacher'] = ''
for s in subjects:
    m = df['weak_subject'] == s
    df.loc[m, 'own_teacher'] = df.loc[m, f'{s}_teacher']
same = df['own_teacher'] == df['best_teacher']
df['mentor'] = np.where(same, df['second_teacher'], df['best_teacher'])
df['mentor_eff'] = np.where(same, df['second_eff'], df['best_eff'])
df.loc[~df['needs_mentor'], 'mentor'] = 'Not needed'
df.loc[~df['needs_mentor'], 'mentor_eff'] = np.nan
print(same[df['needs_mentor']].sum(), 'bachon ka apna teacher hi best tha, unko second best diya')

In [ ]:
reasons = []
for s, u, p, t, e in zip(df['weak_subject'], df['weak_unit'], df['weak_pct'], df['mentor'], df['mentor_eff']):
    reasons.append(f'{s.upper()} unit {u} me {p:.0f}% aaya, us unit me {t} ka efficacy {e} hai')
df['reason'] = reasons
df.loc[~df['needs_mentor'], 'reason'] = 'mentor ki zarurat nahi'
out_cols = ['roll_no', 'full_name', 'class_section', 'cluster_name', 'priority', 'risk', 'weak_subject', 'weak_unit', 'weak_pct', 'own_teacher', 'mentor', 'mentor_eff', 'reason']
assignments = df.loc[df['needs_mentor'], out_cols].sort_values(['priority', 'weak_pct'])
assignments.head(10)

In [ ]:
def get_mentor(roll_no):
    r = df[df['roll_no'].astype(str) == str(roll_no)]
    if r.empty:
        return {'status': 'error', 'message': f'{roll_no} nahi mila'}
    r = r.iloc[0]
    if not r['needs_mentor']:
        return {'status': 'ok', 'roll_no': str(roll_no), 'cluster': r['cluster_name'], 'mentor': 'abhi zarurat nahi'}
    return {'status': 'ok', 'roll_no': str(roll_no), 'name': r['full_name'], 'cluster': r['cluster_name'], 'risk': int(r['risk']),
            'weak_subject': r['weak_subject'], 'weak_unit': int(r['weak_unit']), 'mentor': r['mentor'],
            'mentor_efficacy': float(r['mentor_eff']), 'reason': r['reason']}
for rn in [assignments['roll_no'].iloc[0]] + df[~df['needs_mentor']]['roll_no'].head(1).tolist():
    print(get_mentor(rn))

## 5. Save

In [ ]:
# risk_model.pkl jahan hai wahi save kar dete hain
out_dir = os.path.dirname(risk_path) or '.'
joblib.dump({'kmeans': kmeans, 'scaler': scaler, 'features': cluster_cols, 'cluster_names': cluster_names}, os.path.join(out_dir, 'mentor_cluster_model.pkl'))
t_eff.to_csv(os.path.join(out_dir, 'teacher_efficacy_unitwise.csv'), index=False)
df[out_cols + ['needs_mentor']].to_csv(os.path.join(out_dir, 'mentor_assignments.csv'), index=False)
print(out_dir, os.listdir(out_dir))